# `LocalBootstrap`

`LocalBootstrap` generates *n_bootstraps* resampled datasets of size n.
At every site $i$ the observation placed there is drawn **with replacement**
from all n observations, weighted by a spatial kernel:

$$p_{ij} \propto K\!\left(\frac{d_{ij}}{h}\right)$$

Nearby observations are drawn more often.  Works for spatial data
(GeoDataFrame / (n,2) coordinates) and time-series (1-D time index;
distance = absolute lag).

| Parameter | Role |
|---|---|
| `bandwidth` | Kernel width in CRS units |
| `kernel` | `'gaussian'` (default), `'bisquare'`, `'triangular'`, … |
| `graph` | Pre-built `libpysal.graph.Graph` (overrides bandwidth/kernel) |

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geodatasets

from geovalidate import LocalBootstrap

%matplotlib inline

## Simple example — Chicago community areas

To see clearly how the resampling works we start with the 77 Chicago
community areas.  The variable is **per-capita income** (`PerCInc14`),
which has a strong north–south gradient.

With `bandwidth = 8 km` each area draws mostly from its immediate
neighbours.  The right panel shows **one bootstrap realisation**: the
income value each area received from its draw.  Areas retain roughly
their own neighbourhood's income (local smoothing), but individual areas
may receive a value from a nearby area with a slightly different income.
Because sampling is **with replacement**, a very popular area could
contribute its value to several neighbours simultaneously.

In [ ]:
import geodatasets, numpy as np, geopandas as gpd

chicago = gpd.read_file(geodatasets.get_path("geoda.chicago_health")).to_crs("EPSG:32616")
chicago["income"] = chicago["PerCInc14"].fillna(chicago["PerCInc14"].median())
print(f"n = {len(chicago)} community areas")

In [ ]:
y = chicago["income"].values

lb_simple = LocalBootstrap(n_bootstraps=1, bandwidth=8_000, kernel="gaussian", random_state=42)
indices_simple = next(lb_simple.sample(chicago))
y_boot_simple  = y[indices_simple]

# How many areas drew from themselves (or a unique source)?
print(f"Unique source areas used: {len(set(indices_simple))} / {len(chicago)}")
print(f"(with replacement: some sources may appear more than once)")

In [ ]:
norm = mcolors.Normalize(vmin=y.min(), vmax=y.max())
PT   = dict(norm=norm, cmap="RdYlGn", edgecolor="#333", linewidth=0.5, legend=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.patch.set_facecolor("#f8f8f8")

chicago_p = chicago.copy()

chicago_p["_v"] = y
chicago_p.plot(column="_v", ax=axes[0], **PT)
axes[0].set_title("Original\nPer-capita income (USD)", fontsize=11)
axes[0].set_aspect("equal"); axes[0].axis("off")

chicago_p["_v"] = y_boot_simple
chicago_p.plot(column="_v", ax=axes[1], **PT)
axes[1].set_title("One bootstrap realisation\n(each area shows drawn neighbour's income)", fontsize=11)
axes[1].set_aspect("equal"); axes[1].axis("off")

sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
fig.colorbar(sm, ax=axes, label="Per-capita income (USD)", fraction=0.02, pad=0.02)
fig.suptitle("LocalBootstrap — Chicago community areas  (BW = 8 km, with replacement)", fontsize=12)
fig.tight_layout()
plt.show()

## Larger example — King County house sales

For a dataset with more observations the bootstrap uncertainty map
reveals where price estimates are most sensitive to resampling.

In [ ]:
import geodatasets, numpy as np, geopandas as gpd

gdf_full = gpd.read_file(geodatasets.get_path("geoda.home_sales")).to_crs("EPSG:32610")
gdf_full["log_price"] = np.log(gdf_full["price"])
gdf_full["decile"] = (
    gdf_full["log_price"].rank(pct=True).multiply(10).clip(upper=9.99).astype(int)
)
idx = (
    gdf_full.groupby("decile")
    .apply(lambda g: g.sample(min(60, len(g)), random_state=42), include_groups=False)
    .index.get_level_values(1)
)
gdf = gdf_full.loc[idx].reset_index(drop=True)
print(f"n = {len(gdf)} sales  |  price ${gdf.price.min():,.0f} - ${gdf.price.max():,.0f}")

In [ ]:
N_BOOT = 200
BW     = 10_000

lb = LocalBootstrap(n_bootstraps=N_BOOT, bandwidth=BW, kernel="gaussian", random_state=0)
y_obs = gdf["log_price"].values

boot_samples = np.empty((N_BOOT, len(gdf)))
for b, indices in enumerate(lb.sample(gdf)):
    boot_samples[b] = y_obs[indices]

boot_mean = boot_samples.mean(axis=0)
boot_std  = boot_samples.std(axis=0)
print(f"Std range: {boot_std.min():.3f} – {boot_std.max():.3f} (log USD)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor("#f8f8f8")
PT2  = dict(s=12, alpha=0.8, edgecolors="white", linewidths=0.3)
norm2 = mcolors.Normalize(vmin=y_obs.min(), vmax=y_obs.max())

def scatter_map(ax, vals, title, cmap, norm, label):
    sc = ax.scatter(gdf.geometry.x/1000, gdf.geometry.y/1000,
                    c=vals, cmap=cmap, norm=norm, **PT2)
    plt.colorbar(sc, ax=ax, label=label, fraction=0.04, pad=0.02)
    ax.set_xlabel("Easting (km)"); ax.set_ylabel("Northing (km)")
    ax.set_title(title, fontsize=11); ax.set_aspect("equal")

scatter_map(axes[0], y_obs,   "Observed log(price)",   "RdYlGn", norm2, "log(USD)")
scatter_map(axes[1], boot_mean, f"Bootstrap mean\n(N={N_BOOT}, BW={BW//1000} km)",
            "RdYlGn", norm2, "log(USD)")
scatter_map(axes[2], boot_std,  "Bootstrap std dev\n(higher = more uncertain)",
            "OrRd", mcolors.Normalize(boot_std.min(), boot_std.max()), "std (log USD)")

fig.suptitle(f"LocalBootstrap — King County  (BW={BW//1000} km, N={N_BOOT})", fontsize=13)
fig.tight_layout()
plt.show()